## Run DeepEval LLM-as-Judge with local LLMs of Ollama

In [1]:
from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import ExactMatchMetric;


test_case = LLMTestCase(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    actual_output="The capital of France is Paris."
)

evaluate([test_case], metrics=[
    ExactMatchMetric()
])

✨ You're running DeepEval's latest Exact Match Metric! (using None, strict=False, async_mode=True)...

c:\Users\SANexGenUser\Desktop\USA_Testing\AI\deepeval-llm-evaluation\.venv\Lib\site-packages\rich\live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Exact Match (score: 1.0, threshold: 1.0, strict: False, evaluation model: None, reason: The actual and expected outputs are exact matches., error: None)

For test case:

  - input: What is the capital of France?
  - actual output: The capital of France is Paris.
  - expected output: The capital of France is Paris.
  - context: None
  - retrieval context: None


Overall Metric Pass Rates

Exact Match: 100.00% pass rate




⚠ WARNING: No hyperparameters logged.
» ]8;id=7475217;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.2s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Exact Match', threshold=1.0, success=True, score=1.0, reason='The actual and expected outputs are exact matches.', strict_mode=False, evaluation_model=None, error=None, evaluation_cost=None, verbose_logs=None)], conversational=False, multimodal=False, input='What is the capital of France?', actual_output='The capital of France is Paris.', expected_output='The capital of France is Paris.', context=None, retrieval_context=None, turns=None, additional_metadata=None)], confident_link=None, test_run_id=None)

In [1]:
!deepeval set-ollama --model llama3.2:latest

🙌 Congratulations! You're now using a local Ollama model `llama3.2:latest` for
all evals that require an LLM.


## Test Code LLM-as-a-judge - AnswerRelevancy - Thinking Mode

In [ ]:
from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import AnswerRelevancyMetric;
from dotenv import load_dotenv, find_dotenv;
from deepeval.evaluate import AsyncConfig;
from deepeval.models import OllamaModel;
import os

load_dotenv(find_dotenv())

ollama_model = OllamaModel(model=os.getenv("LOCAL_OLLAMA_MODEL"), base_url=os.getenv("LOCAL_OLLAMA_URL"))

test_case = LLMTestCase(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    actual_output="Paris."
)

evaluate(test_cases =[test_case], 
         metrics=[AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using deepseek-r1:1.5b (Ollama), strict=False, 
async_mode=False)...



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: deepseek-r1:1.5b (Ollama), reason: Everything is relevant and correct for addressing your question about the capital of France!, error: None)

For test case:

  - input: What is the capital of France?
  - actual output: Paris.
  - expected output: The capital of France is Paris.
  - context: None
  - retrieval context: None


Overall Metric Pass Rates

Answer Relevancy: 100.00% pass rate




⚠ WARNING: No hyperparameters logged.
» ]8;id=12701319;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 127.55s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='Everything is relevant and correct for addressing your question about the capital of France!', strict_mode=False, evaluation_model='deepseek-r1:1.5b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Statements:\n[\n    ""\n] \n \nVerdicts:\n[]')], conversational=False, multimodal=False, input='What is the capital of France?', actual_output='Paris.', expected_output='The capital of France is Paris.', context=None, retrieval_context=None, turns=None, additional_metadata=None)], confident_link=None, test_run_id=None)

## Test Code LLM-as-a-judge - AnswerRelevancy - No Thinking Mode

In [3]:
from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import AnswerRelevancyMetric;
from dotenv import load_dotenv, find_dotenv;
from deepeval.evaluate import AsyncConfig;
from deepeval.models import OllamaModel;
import os
from typing import Tuple, Union, Optional
from pydantic import BaseModel

load_dotenv(find_dotenv())


class OllamaModelNoThink(OllamaModel):
    def generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model()
        messages = [{"role": "user", "content": prompt}]

        response = chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )
    
    async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model(async_mode=True)
        messages = [{"role": "user", "content": prompt}]

        response = await chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

ollama_model = OllamaModelNoThink(model=os.getenv("LOCAL_OLLAMA_MODEL"), base_url=os.getenv("LOCAL_OLLAMA_URL"))

test_case = LLMTestCase(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    actual_output="Bengaluru."
)

evaluate(test_cases =[test_case], 
         metrics=[AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using deepseek-r1:1.5b (Ollama), strict=False, 
async_mode=False)...



Metrics Summary

  - ✅ Answer Relevancy (score: 0.5, threshold: 0.5, strict: False, evaluation model: deepseek-r1:1.5b (Ollama), reason: The answer relevance score is 0.50 because the statement 'the capital of France' is directly relevant to the question asked and does not contain any irrelevant information., error: None)

For test case:

  - input: What is the capital of France?
  - actual output: Bengaluru.
  - expected output: The capital of France is Paris.
  - context: None
  - retrieval context: None


Overall Metric Pass Rates

Answer Relevancy: 100.00% pass rate




⚠ WARNING: No hyperparameters logged.
» ]8;id=12701321;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 56.96s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=0.5, reason="The answer relevance score is 0.50 because the statement 'the capital of France' is directly relevant to the question asked and does not contain any irrelevant information.", strict_mode=False, evaluation_model='deepseek-r1:1.5b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Statements:\n[\n    "Bengaluru.",\n    "The city is known for its vibrant culture and diverse history."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": "The statement provides relevant information about the location."\n    },\n    {\n        "verdict": "no",\n        "reason": "The statement is irrelevant to addressing the input."\n    }\n]')], conversational=False, multimodal=False, input='What is the capital of France?', actual_output='Bengaluru.', expected_output='The capital of France is Paris.', context